In [ ]:
import fitz  # PyMuPDF
import re
import os
import tkinter as tk
from tkinter import filedialog, messagebox, ttk

# --- CONFIGURAÇÕES DE UI ---
class DialogoOpcoes(tk.Toplevel):
    def __init__(self, parent):
        super().__init__(parent)
        self.title("Configurações de Anonimização")
        self.geometry("400x350")
        self.resultado = None

        # Variáveis de controle
        self.vars = {
            "cpf": tk.BooleanVar(value=True),
            "rg": tk.BooleanVar(value=True),
            "email_tel": tk.BooleanVar(value=True),
            "doc_sei": tk.BooleanVar(value=True), # Código 7 dígitos
            "qr_code": tk.BooleanVar(value=True), # QR Code e Barras Laterais
            "links": tk.BooleanVar(value=True)    # Remover hiperlinks
        }

        # Layout
        lbl = ttk.Label(self, text="Selecione os elementos para anonimizar:", font=("Arial", 10, "bold"))
        lbl.pack(pady=10)

        opcoes = [
            ("Mascarar CPFs (***.123.456-**)", "cpf"),
            ("Mascarar RGs e Datas (RG: XXXXXX)", "rg"),
            ("Mascarar E-mails e Telefones", "email_tel"),
            ("Mascarar Nº Documento/Verificador (XXXXX11)", "doc_sei"),
            ("Mascarar QR Codes e Barras Laterais (SEI)", "qr_code"),
            ("Remover todos os Links Clicáveis", "links")
        ]

        for texto, chave in opcoes:
            chk = ttk.Checkbutton(self, text=texto, variable=self.vars[chave])
            chk.pack(anchor="w", padx=20, pady=2)

        btn_frame = ttk.Frame(self)
        btn_frame.pack(pady=20)
        
        btn_ok = ttk.Button(btn_frame, text="Confirmar", command=self.confirmar)
        btn_ok.pack(side="left", padx=10)
        
        btn_cancel = ttk.Button(btn_frame, text="Cancelar", command=self.cancelar)
        btn_cancel.pack(side="right", padx=10)

    def confirmar(self):
        self.resultado = {k: v.get() for k, v in self.vars.items()}
        self.destroy()

    def cancelar(self):
        self.resultado = None
        self.destroy()

# --- FUNÇÕES DE MÁSCARA ---
def mascarar_cpf_fmt(cpf_original):
    numeros = re.sub(r'\D', '', cpf_original)
    if len(numeros) == 11:
        return f"***.{numeros[3:6]}.{numeros[6:9]}-**"
    return "***.***.***-**"

def mascarar_doc_7_digitos_fmt(numero):
    if len(numero) == 7:
        return "XXXXX" + numero[-2:]
    return "XXXXXXX"

def processar_arquivo(caminho_entrada, opcoes):
    try:
        pasta_pai = os.path.dirname(caminho_entrada)
        nome_arquivo = os.path.basename(caminho_entrada)
        
        # Salva na MESMA pasta com sufixo _pb
        caminho_saida = os.path.join(pasta_pai, os.path.splitext(nome_arquivo)[0] + "_pb.pdf")

        doc = fitz.open(caminho_entrada)
        
        # Regex
        regex_cpf = re.compile(r'\b\d{3}\.\d{3}\.\d{3}-\d{2}\b')
        regex_rg_contexto = re.compile(r'(RG:)\s*([0-9.-]+\s?\d*)', re.IGNORECASE)
        regex_data_exp = re.compile(r'(Data de Expedição:)\s*(\d{2}/\d{2}/\d{4})', re.IGNORECASE)
        regex_7_digitos = re.compile(r'\b\d{7}\b')
        regex_crc = re.compile(r'\b[0-9A-F]{8}\b', re.IGNORECASE)
        regex_email = re.compile(r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b')
        regex_telefone = re.compile(r'(?:\(?\d{2}\)?\s?)?(?:9\d{4}|\d{4})[-\s]\d{4}')

        print(f"Processando: {nome_arquivo}...")

        for page in doc:
            texto_pagina = page.get_text()
            altura_pagina = page.rect.height
            largura_pagina = page.rect.width

            # --- CORREÇÃO: LINKS (Compatível com versões antigas) ---
            if opcoes["links"]:
                if hasattr(page, "delete_links"):
                    # Método moderno
                    page.delete_links()
                else:
                    # Método legado (itera e deleta um por um)
                    for link in page.get_links():
                        page.delete_link(link)

            # --- OPÇÃO: QR CODES E CÓDIGO DE BARRAS (Estratégia Espacial SEI) ---
            if opcoes["qr_code"]:
                # 1. Barra Lateral Direita (Padrão SEI - Código de Barras Vertical)
                # Define uma faixa de 25 pixels na borda direita
                rect_barra_lateral = fitz.Rect(largura_pagina - 25, 0, largura_pagina, altura_pagina)
                annot = page.add_redact_annot(rect_barra_lateral, fill=(0, 0, 0)) # Tarja preta
                annot.update()

                # 2. QR Code próximo ao bloco de autenticação
                areas_auth = page.search_for("A autenticidade do documento pode ser conferida no site")
                if areas_auth:
                    y_base = min([r.y0 for r in areas_auth])
                    # O QR Code no SEI geralmente fica à esquerda ou acima desse texto.
                    for area in areas_auth:
                        # Tenta cobrir um quadrado à esquerda do texto
                        rect_qr_left = fitz.Rect(area.x0 - 60, area.y0 - 10, area.x0, area.y1 + 40)
                        annot = page.add_redact_annot(rect_qr_left, fill=(1, 1, 1)) # Branco para "apagar"
                        annot.update()

            # --- OPÇÃO: E-MAIL E TELEFONE ---
            if opcoes["email_tel"]:
                for match in regex_email.finditer(texto_pagina):
                    areas = page.search_for(match.group())
                    for area in areas:
                        annot = page.add_redact_annot(area, text="[E-MAIL]", fontname="Helvetica", fontsize=8, align=0, fill=(1,1,1))
                        annot.update()

                for match in regex_telefone.finditer(texto_pagina):
                    fone = match.group()
                    if re.match(r'^(19|20)\d{2}-\d{4}', fone): continue
                    
                    areas = page.search_for(fone)
                    for area in areas:
                        annot = page.add_redact_annot(area, text="[TEL]", fontname="Helvetica", fontsize=8, align=0, fill=(1,1,1))
                        annot.update()

            # --- OPÇÃO: CPF ---
            if opcoes["cpf"]:
                for match in regex_cpf.finditer(texto_pagina):
                    cpf = match.group()
                    areas = page.search_for(cpf)
                    mascara = mascarar_cpf_fmt(cpf)
                    for area in areas:
                        annot = page.add_redact_annot(area, text=mascara, fontname="Helvetica", fontsize=10, align=0, fill=(1,1,1))
                        annot.update()

            # --- OPÇÃO: RG ---
            if opcoes["rg"]:
                for match in regex_rg_contexto.finditer(texto_pagina):
                    areas = page.search_for(match.group(0))
                    for area in areas:
                        annot = page.add_redact_annot(area, text="RG: XXXXXX", fontname="Helvetica", fontsize=10, align=0, fill=(1,1,1))
                        annot.update()
                
                for match in regex_data_exp.finditer(texto_pagina):
                    areas = page.search_for(match.group(0))
                    for area in areas:
                        annot = page.add_redact_annot(area, text="Data de Expedição: XX/XX/XXXX", fontname="Helvetica", fontsize=10, align=0, fill=(1,1,1))
                        annot.update()

            # --- OPÇÃO: DOC SEI (7 DÍGITOS) ---
            if opcoes["doc_sei"]:
                limite_y_corte = altura_pagina * 0.85
                areas_assinatura = page.search_for("Documento assinado eletronicamente por")
                if areas_assinatura:
                    limite_y_corte = min([r.y0 for r in areas_assinatura])

                for match in regex_7_digitos.finditer(texto_pagina):
                    numero = match.group()
                    areas_num = page.search_for(numero)
                    for area in areas_num:
                        if area.y0 > limite_y_corte:
                            mascara_parcial = mascarar_doc_7_digitos_fmt(numero)
                            annot = page.add_redact_annot(area, text=mascara_parcial, fontname="Helvetica", fontsize=9, align=1, fill=(1,1,1))
                            annot.update()
                
                areas_auth = page.search_for("A autenticidade do documento pode ser conferida no site")
                if areas_auth:
                    y_auth = min([r.y0 for r in areas_auth])
                    for match_crc in regex_crc.finditer(texto_pagina):
                        crc = match_crc.group()
                        areas_crc = page.search_for(crc)
                        for area in areas_crc:
                            if area.y0 >= y_auth:
                                annot = page.add_redact_annot(area, text="XXXXXXXX", fontname="Helvetica", fontsize=8, align=1, fill=(0,0,0))
                                annot.update()

            # APLICA AS REDAÇÕES
            page.apply_redactions()

        doc.save(caminho_saida, garbage=4, deflate=True)
        doc.close()
        return True

    except Exception as e:
        print(f"Erro em {caminho_entrada}: {e}")
        return False

def main():
    root = tk.Tk()
    root.withdraw()

    dialogo = DialogoOpcoes(root)
    root.wait_window(dialogo)
    
    opcoes_escolhidas = dialogo.resultado
    if not opcoes_escolhidas:
        print("Operação cancelada pelo usuário.")
        return

    resposta = messagebox.askyesno("Seleção de Origem", 
                                   "Deseja processar uma PASTA inteira?\n\n"
                                   "Sim = Pasta (Recursivo)\nNão = Arquivo Único")

    arquivos_para_processar = []
    
    if resposta: 
        pasta_selecionada = filedialog.askdirectory(title="Selecione a Pasta Raiz")
        if not pasta_selecionada: return
        
        for raiz, dirs, arqs in os.walk(pasta_selecionada):
            for arq in arqs:
                if arq.lower().endswith(".pdf") and "_pb" not in arq:
                    arquivos_para_processar.append(os.path.join(raiz, arq))
    else:
        arquivo = filedialog.askopenfilename(title="Selecione o PDF", filetypes=[("PDF", "*.pdf")])
        if not arquivo: return
        arquivos_para_processar.append(arquivo)

    if not arquivos_para_processar:
        messagebox.showinfo("Info", "Nenhum arquivo PDF encontrado.")
        return

    contador = 0
    total = len(arquivos_para_processar)
    print(f"Iniciando processamento de {total} arquivos...\n")
    
    for caminho in arquivos_para_processar:
        if processar_arquivo(caminho, opcoes_escolhidas):
            contador += 1
            
    messagebox.showinfo("Concluído", f"Processamento finalizado!\n{contador} arquivos gerados com sucesso.")



In [2]:
main()

Iniciando processamento de 46 arquivos...

Processando: E-mail implementação bolsas - FACEPE.pdf...
Processando: Ofício FACEPE - 2024.pdf...
Processando: RESULTADO FINAL BOLSAS DE PRODUTIVIDADE DO CNPq - FACEPE.pdf...
Processando: SEI_CNPq - 2032007 - Acordo de Cooperação - FACEPE.pdf...
Processando: SEI_CNPq - 2032015 - Plano de Trabalho - FACEPE.pdf...
Processando: Autorizacao_Bolsa_de_Produtividade_Estadual_CNPq_FAPEMIG.pdf...
Processando: Ofício FAPEMIG- 2024.pdf...
Processando: Relação Bolsas implementadas Acordo de Cooperação FAPEMIG - 2024.pdf...
Processando: SEI_CNPq - 2100712 - Acordo de Cooperação - FAPEMIG.pdf...
Processando: SEI_CNPq - 2137123 - Plano de Trabalho - FAPEMIG.pdf...
Processando: E-mail implementação bolsas - FUNDECT.pdf...
Processando: Ofício FUNDECT - 2024.pdf...
Processando: SEI_CNPq - 2018811 - Acordo de Cooperação - FUNDECT 2024.pdf...
Processando: SEI_CNPq - 2018838 - Plano de Trabalho - FUNDECT 2024.pdf...
Processando: E-mail implementação bolsas - FAPEA